# Stage 4: PDF Text Extraction

This notebook loads the sample Networking PDF and checks that PyMuPDF can extract its selectable text. In VS Code, choose the `Python (AI IT Exam Question Generator)` kernel before running the cells.

In [4]:
from pathlib import Path
import sys

# Find the project whether Jupyter starts in the project or notebooks folder.
search_roots = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "ai_it_exam_question_generator"]
project_root = next(
    (root for root in search_roots if (root / "src" / "pdf_processing.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not find the project src/pdf_processing.py file.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.pdf_processing import extract_pdf_text

print(f"Project folder: {project_root}")

Project folder: C:\Users\user\Documents\CAT\ai_it_exam_question_generator


In [5]:
sample_pdf_path = project_root / "data" / "sample" / "networking_fundamentals.pdf"
extracted_text = extract_pdf_text(sample_pdf_path)

print(f"PDF: {sample_pdf_path.name}")
print(f"Extracted characters: {len(extracted_text)}")
print("\nText preview:\n")
print(extracted_text[:500])

PDF: networking_fundamentals.pdf
Extracted characters: 2951

Text preview:

Networking Fundamentals
A computer network is a group of connected devices that exchange data and share resources. A Local
Area Network (LAN) covers a limited area such as a home, office, or campus. A Wide Area Network
(WAN) spans a larger geographic area and can connect multiple LANs.
A network switch connects devices within a LAN. It forwards Ethernet frames using the destination
device's MAC address. A router connects different networks and forwards IP packets using IP
addresses and routing i


In [3]:
assert len(extracted_text) > 0
assert "DHCP" in extracted_text
assert "TCP" in extracted_text
print("PASS: text was extracted and expected Networking terms were found.")

PASS: text was extracted and expected Networking terms were found.


## PDF text and OCR

PyMuPDF first extracts selectable text. For pages that contain only images, it attempts local OCR through Tesseract. On Windows, install Tesseract with English language data and add it to `PATH`. If Tesseract is missing, image-only pages show installation guidance. A completely blank page still has no usable text.

## Stage 5: Clean and split the notes

PDF extraction often contains extra line breaks. We normalize whitespace, then split the cleaned text into overlapping chunks. Overlap preserves nearby context around boundaries. Each chunk is kept under the selected character limit.

In [ ]:
from src.text_processing import clean_text, split_text_into_chunks

cleaned_text = clean_text(extracted_text)
chunks = split_text_into_chunks(cleaned_text, max_chars=700, overlap=100)

print(f"Cleaned characters: {len(cleaned_text)}")
print(f"Chunks created: {len(chunks)}")
for index, chunk in enumerate(chunks, start=1):
    print(f"Chunk {index}: {len(chunk)} characters")

assert cleaned_text
assert len(chunks) > 1
assert all(len(chunk) <= 700 for chunk in chunks)
print("PASS: cleaned notes were split into size-limited chunks.")

## Stage 6: Retrieve relevant notes locally

The app uses TF-IDF, a local text-matching method from scikit-learn, to rank note chunks that contain words from the chosen learning unit. It does not download an embedding model or send notes to an API. This is a local RAG-style workflow: find relevant passages first, then build quiz items directly from definitions and statements in those passages. The generator is template-based rather than a free-form language model.

In [ ]:
from src.offline_generation import generate_offline_questions, retrieve_relevant_chunks

retrieved_notes = retrieve_relevant_chunks(
    chunks,
    subject="Networking",
    topic_unit="Domain Name System DNS",
    top_k=3,
)
local_questions = generate_offline_questions(
    retrieved_notes,
    subject="Networking",
    topic_unit="Domain Name System DNS",
    difficulty="Easy",
    count=2,
    question_type="Multiple choice",
)

for question in local_questions:
    print(question["question"])
    for label, option in question["options"].items():
        print(f"{label}. {option}")
    print(f"Source: {question['source']}\n")

assert len(local_questions) == 2
assert all(set(question["options"]) == {"A", "B", "C", "D"} for question in local_questions)
print("PASS: local TF-IDF retrieved relevant notes and built questions with no API.")